# 🧹 Notebook 02 — Text Preprocessing

**Goal:** Bersihin tweet biar siap dipake buat training model.

## Kenapa preprocessing penting?
Tweet itu **kotor banget**:
- URL: `http://t.co/abc123`
- Mention: `@VirginAmerica`
- Hashtag: `#delayed`
- Emoji & special chars
- Singkatan: "u", "im", "dont"
- Inconsistent case: "FLIGHT" vs "flight"

Model bakal kacau kalo dikasih input mentah. Garbage in → garbage out.

## Yang bakal kita lakuin:
1. Load data dari notebook 01
2. Tampilin contoh tweet kotor
3. Build cleaning function step-by-step
4. Apply cleaning ke semua tweet
5. Tokenization & stopword removal
6. Bandingin tweet sebelum vs sesudah
7. Save clean data buat modeling

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download NLTK data (sekali aja, nanti dia skip kalo udah ada)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

pd.set_option('display.max_colwidth', 200)
print("✅ Libraries loaded!")

✅ Libraries loaded!


## 2. Load Data dari Notebook 01

In [2]:
df = pd.read_csv('../data/processed/tweets_eda.csv')

# Ambil kolom yang dibutuhin aja
df = df[['tweet_id', 'airline_sentiment', 'airline', 'text', 'negativereason', 'retweet_count', 'tweet_created']].copy()

print(f"Shape: {df.shape}")
df.head(3)

Shape: (14640, 7)


,tweet_id,airline_sentiment,airline,text,negativereason,retweet_count,tweet_created
0,570306133677760513,neutral,Virgin America,@VirginAmerica What @dhepburn said.,NaN,0,2015-02-24 11:35:52 -0800
1,570301130888122368,positive,Virgin America,@VirginAmerica plus you've added commercials to the experience... tacky.,NaN,0,2015-02-24 11:15:59 -0800
2,570301083672813571,neutral,Virgin America,@VirginAmerica I didn't today... Must mean I need to take another trip!,NaN,0,2015-02-24 11:15:48 -0800


## 3. Liat Contoh Tweet "Kotor"

Sebelum cleaning, perhatiin pattern apa aja yang harus dibersihin.

In [3]:
# Sample beberapa tweet biar keliatan masalahnya
sample_tweets = df['text'].sample(10, random_state=7).tolist()
for i, tweet in enumerate(sample_tweets, 1):
    print(f"{i}. {tweet}")
    print()

1. @united you've been officially displaced by @DeltaAssist for better flight experience, pleasant cabin crew and timeliness. UA 1m switching!

2. @united Are the current on board food menus available anywhere online?

3. @AmericanAir yes. Shows Cancelled Flightled &amp; asks for her to contact you at 800 number.

4. @JetBlue it's poop and barely works

5. @JetBlue thanks so much for help Us, u r amazing!

6. @united Second day &amp; attempt trying to leave IAD &amp; I'm on another plane delayed for mechanical issues. Over an hour delayed!

7. @VirginAmerica You should still develop an app - then you will be my favorite airline.

8. @united please explain why I need to pay bag fees twice, equally $1200, because a delayed flight resulted in bags being rechecked overnight?

9. @USAirways , since jan 22 have been on 10 flights. Why the disrespect in CLT, flt #2063 at 2:50pm on 18feb to DCA? just asked herba question

10. @USAirways You guys are screwing up my trip. Thanks for nothing. #la

**Yang kita liat:**
- 🔗 URL: `http://t.co/...`
- 👤 Mention: `@VirginAmerica`, `@united`
- 🔣 Karakter aneh: `&amp;`, `RT`, ekstra spasi
- 📝 Mix case: kombinasi besar/kecil
- 🔤 Punctuation berlebihan: `!!!`, `?????`
- 😀 Emoji & symbol

## 4. Build Cleaning Function (Step-by-Step)

Kita bangun function bertahap, biar paham tiap step ngapain.

In [4]:
# Test pake 1 tweet kotor
test_tweet = "@VirginAmerica WHAT a TERRIBLE flight!!! Delayed AGAIN... check this http://t.co/abc123 #worstairline RT &amp; please respond :("
print(f"ORIGINAL:\n{test_tweet}\n")

ORIGINAL:
@VirginAmerica WHAT a TERRIBLE flight!!! Delayed AGAIN... check this http://t.co/abc123 #worstairline RT &amp; please respond :(



In [5]:
# Step 1: Lowercase
step1 = test_tweet.lower()
print(f"After lowercase:\n{step1}\n")

After lowercase:
@virginamerica what a terrible flight!!! delayed again... check this http://t.co/abc123 #worstairline rt &amp; please respond :(



In [6]:
# Step 2: Remove URLs
step2 = re.sub(r'http\S+|www\S+|https\S+', '', step1)
print(f"After URL removal:\n{step2}\n")

After URL removal:
@virginamerica what a terrible flight!!! delayed again... check this  #worstairline rt &amp; please respond :(



In [7]:
# Step 3: Remove mentions (@username)
step3 = re.sub(r'@\w+', '', step2)
print(f"After mention removal:\n{step3}\n")

After mention removal:
 what a terrible flight!!! delayed again... check this  #worstairline rt &amp; please respond :(



In [8]:
# Step 4: Remove hashtag symbols (tapi keep kata-nya)
# '#worstairline' jadi 'worstairline' (kata ini punya info sentiment!)
step4 = re.sub(r'#', '', step3)
print(f"After hashtag handling:\n{step4}\n")

After hashtag handling:
 what a terrible flight!!! delayed again... check this  worstairline rt &amp; please respond :(



In [9]:
# Step 5: Replace HTML entities
step5 = step4.replace('&amp;', 'and').replace('&lt;', '').replace('&gt;', '').replace('&quot;', '')
print(f"After HTML entities:\n{step5}\n")

After HTML entities:
 what a terrible flight!!! delayed again... check this  worstairline rt and please respond :(



In [10]:
# Step 6: Remove special chars, keep only letters & spaces
step6 = re.sub(r'[^a-z\s]', ' ', step5)
print(f"After special char removal:\n{step6}\n")

After special char removal:
 what a terrible flight    delayed again    check this  worstairline rt and please respond   



In [11]:
# Step 7: Remove extra whitespace
step7 = re.sub(r'\s+', ' ', step6).strip()
print(f"After whitespace cleanup:\n{step7}\n")

After whitespace cleanup:
what a terrible flight delayed again check this worstairline rt and please respond



**Bandingin:**
- Sebelum: `@VirginAmerica WHAT a TERRIBLE flight!!! Delayed AGAIN... check this http://t.co/abc123 #worstairline RT &amp; please respond :(`
- Sesudah: `what a terrible flight delayed again check this worstairline rt and please respond`

Jauh lebih clean! 🧹

## 5. Build Final Cleaning Function

Sekarang gabungin semua step ke 1 function biar reusable.

In [12]:
def clean_text(text):
    """
    Bersihin tweet text dari noise.
    
    Steps:
    1. Lowercase
    2. Remove URLs
    3. Remove mentions (@user)
    4. Remove hashtag symbol (keep word)
    5. Replace HTML entities
    6. Remove special chars (keep letters & spaces)
    7. Remove extra whitespace
    """
    if not isinstance(text, str):
        return ''
    
    text = text.lower()                                          # lowercase
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)         # URLs
    text = re.sub(r'@\w+', '', text)                             # mentions
    text = re.sub(r'#', '', text)                                # hashtag symbol
    text = text.replace('&amp;', 'and').replace('&lt;', '')      # HTML entities
    text = text.replace('&gt;', '').replace('&quot;', '')
    text = re.sub(r'[^a-z\s]', ' ', text)                        # special chars
    text = re.sub(r'\s+', ' ', text).strip()                     # extra whitespace
    
    return text

# Test
print(clean_text(test_tweet))

what a terrible flight delayed again check this worstairline rt and please respond


## 6. Apply Cleaning ke Semua Tweet

In [13]:
df['text_clean'] = df['text'].apply(clean_text)

# Lihat hasil sebelum vs sesudah
comparison = df[['text', 'text_clean']].sample(5, random_state=42)
for idx, row in comparison.iterrows():
    print(f"📝 ORIGINAL: {row['text']}")
    print(f"✨ CLEANED:  {row['text_clean']}")
    print("-" * 100)

📝 ORIGINAL: @SouthwestAir you're my early frontrunner for best airline! #oscars2016
✨ CLEANED:  you re my early frontrunner for best airline oscars
----------------------------------------------------------------------------------------------------
📝 ORIGINAL: @USAirways how is it that my flt to EWR was Cancelled Flightled yet flts to NYC from USAirways are still flying?
✨ CLEANED:  how is it that my flt to ewr was cancelled flightled yet flts to nyc from usairways are still flying
----------------------------------------------------------------------------------------------------
📝 ORIGINAL: @JetBlue what is going on with your BDL to DCA flights yesterday and today?! Why is every single one getting delayed?
✨ CLEANED:  what is going on with your bdl to dca flights yesterday and today why is every single one getting delayed
----------------------------------------------------------------------------------------------------
📝 ORIGINAL: @JetBlue do they have to depart from Washington, D.

## 7. Tokenization & Stopword Removal

**Tokenization:** Pecah teks jadi list of words.  
**Stopword removal:** Buang kata-kata umum (the, is, a, an, dll) yang gak bawa info sentiment.

⚠️ **Catatan penting:** Untuk **VADER & HuggingFace**, kita pake `text_clean` (TANPA stopword removal). Karena model itu butuh konteks kalimat utuh. Kita cuma butuh `text_tokens` (TANPA stopwords) buat **Naive Bayes & Logistic Regression**.

In [14]:
# Setup stopwords
stop_words = set(stopwords.words('english'))

# Kita JANGAN drop kata negasi karena penting buat sentiment!
# 'not good' beda banget sama 'good'
negation_words = {'not', 'no', 'nor', 'neither', 'never', "n't", 'nothing', 'nobody'}
stop_words = stop_words - negation_words

print(f"Total stopwords: {len(stop_words)}")
print(f"Contoh stopwords: {list(stop_words)[:20]}")
print(f"\nKata negasi yang KITA PERTAHANKAN: {negation_words}")

Total stopwords: 195
Contoh stopwords: ['who', 'by', 'been', 'didn', 'at', 'having', 'his', 'have', 'for', "isn't", 'was', 'after', "i'd", 'because', 'had', "she's", 'be', "they'd", 'further', 'now']

Kata negasi yang KITA PERTAHANKAN: {'nobody', 'no', 'nor', 'not', 'neither', "n't", 'nothing', 'never'}


In [15]:
def tokenize_and_remove_stopwords(text):
    """Tokenize + buang stopwords (kecuali negasi)."""
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return tokens

# Bikin kolom baru: tokens
df['tokens'] = df['text_clean'].apply(tokenize_and_remove_stopwords)

# Bikin kolom text_tokens: join tokens back to string (buat input ke ML model)
df['text_tokens'] = df['tokens'].apply(lambda x: ' '.join(x))

# Compare
sample = df[['text_clean', 'text_tokens']].sample(5, random_state=42)
for idx, row in sample.iterrows():
    print(f"BEFORE: {row['text_clean']}")
    print(f"AFTER:  {row['text_tokens']}")
    print("-" * 100)

BEFORE: you re my early frontrunner for best airline oscars
AFTER:  early frontrunner best airline oscars
----------------------------------------------------------------------------------------------------
BEFORE: how is it that my flt to ewr was cancelled flightled yet flts to nyc from usairways are still flying
AFTER:  flt ewr cancelled flightled yet flts nyc usairways still flying
----------------------------------------------------------------------------------------------------
BEFORE: what is going on with your bdl to dca flights yesterday and today why is every single one getting delayed
AFTER:  going bdl dca flights yesterday today every single one getting delayed
----------------------------------------------------------------------------------------------------
BEFORE: do they have to depart from washington d c
AFTER:  depart washington
----------------------------------------------------------------------------------------------------
BEFORE: i can probably find some of the

## 8. Quality Check

In [16]:
# Cek tweet yang jadi kosong setelah cleaning
empty_clean = df[df['text_clean'].str.len() == 0]
empty_tokens = df[df['text_tokens'].str.len() == 0]

print(f"Tweet dengan text_clean kosong: {len(empty_clean)}")
print(f"Tweet dengan text_tokens kosong: {len(empty_tokens)}")

# Stats panjang
print(f"\nAvg length text_clean: {df['text_clean'].str.len().mean():.1f} chars")
print(f"Avg length text_tokens: {df['text_tokens'].str.len().mean():.1f} chars")
print(f"Avg jumlah tokens: {df['tokens'].apply(len).mean():.1f} words")

Tweet dengan text_clean kosong: 0
Tweet dengan text_tokens kosong: 24

Avg length text_clean: 84.6 chars
Avg length text_tokens: 56.6 chars
Avg jumlah tokens: 8.7 words


In [17]:
# Drop tweet yang kosong setelah cleaning (gak bisa di-modeling)
before = len(df)
df = df[df['text_tokens'].str.len() > 0].reset_index(drop=True)
after = len(df)

print(f"Drop {before - after} tweet kosong")
print(f"Final shape: {df.shape}")

Drop 24 tweet kosong
Final shape: (14616, 10)


## 9. Save Data Bersih

Kita save ke `data/processed/tweets_clean.csv` untuk dipake di notebook modeling.

In [18]:
# Drop kolom tokens (list) — CSV gak support list, kita pake text_tokens (string) aja
df_save = df.drop(columns=['tokens'])

df_save.to_csv('../data/processed/tweets_clean.csv', index=False)
print(f"✅ Saved to data/processed/tweets_clean.csv")
print(f"   Shape: {df_save.shape}")
print(f"   Columns: {list(df_save.columns)}")

✅ Saved to data/processed/tweets_clean.csv
   Shape: (14616, 9)
   Columns: ['tweet_id', 'airline_sentiment', 'airline', 'text', 'negativereason', 'retweet_count', 'tweet_created', 'text_clean', 'text_tokens']


## 10. Recap

### 🧹 Apa yang udah kita lakuin:
- Lowercase semua text
- Remove URL, mention, hashtag symbol, HTML entities
- Remove special chars & extra whitespace
- Tokenization
- Remove stopwords (tapi **keep negasi** — penting buat sentiment!)

### 📁 Output:
- `data/processed/tweets_clean.csv` — siap dipake buat modeling

### 🚀 Next: Notebook 03 — Modeling

Kita bakal training 4 model:
1. **VADER** — rule-based, pakai `text_clean`
2. **Naive Bayes** — klasik ML, pakai `text_tokens`
3. **Logistic Regression** — klasik ML, pakai `text_tokens`
4. **HuggingFace (DistilBERT)** — deep learning pre-trained, pakai `text_clean`

Habis itu kita bandingin akurasi-nya di Notebook 04!